In [5]:
import numpy as np
from typing import Any
from lf_toolkit.evaluation import Result, Params

In [6]:
import os
import json

cwd = os.getcwd()
dir = os.path.dirname(cwd)
reference_path = os.path.join(dir, "data", "referenceMIDI.json")
response_path = os.path.join(dir, "data", "responseMIDI.json")

with open(reference_path) as f1:
    reference = json.load(f1)

with open(response_path) as f2:
    response = json.load(f2)

# Dynamic Time Warping

The goal of DTW is to find an optimal possibly nonlinear alignment between response MIDI sequence to reference MIDI sequence.

The algorithm can be found in this book (Chpater 3.2 Dynamic Time Warping):
M. Müller, Fundamentals of Music Processing. Cham: Springer International Publishing, 2021, ISBN: 9783030698072. DOI:https://doi.org/10.1007/978-3-030-69808-9.


Basic approach:
- Evaluating the local cost measure for each pair of elements in the response(X) and reference(Y) sequences. 
- Dynamic programming to find an alignment path between X and Y having minimal overall cost, i.e. DTW distance. The algorithm computes a cumulative distance path, the timestamps of the target MIDI are warped so they perfectly align with the anchor points of the reference MIDI.

However, this basic approach will not correctly handle the missing note case as expected, because it allows a note to match with multiple notes. Let's say, there is a note missing in the response, this algorithm tends to match a response note with two reference note, instead of reporting the missing problem.

! need to modify this algorithm to make it handle the missing/extra problem correctly.

In [7]:
def compute_cost(note1, note2):
    """
    Compute the local cost measure for each pair of notes.
 
    Only pitch is involved in the cost calculation, since the purpose 
    is to check if the user played missing or extra notes.
 
    Args:
        note1: dict with keys "pitch" (int), "start" (float), "duration" (float)
        note2: dict with keys "pitch" (int), "start" (float), "duration" (float)
 
    Returns:
        int: cost value >= 0 (lower means more similar pitch)
    """
    return int(abs(note1["pitch"] - note2["pitch"]))


def cost_matrix(response_notes, ref_notes):
    """
    Build the local cost matrix C of size (N x M).
    C[i, j] = note_cost(ref_notes[i], response_notes[j])
 
    Args:
        response_notes: list of dicts, each with keys "pitch", "start", "duration"
        ref_notes: list of dicts, each with keys "pitch", "start", "duration"
 
    Returns:
        numpy array of shape (N, M) - cost matrix where C[i,j] is the cost of 
        aligning response_notes[i] with ref_notes[j]
    """
    N = len(response_notes)
    M = len(ref_notes)
    C= np.zeros((N, M))
 
    for i in range(N):
        for j in range(M):
            C[i, j] = compute_cost(response_notes[i], ref_notes[j])
    return C


def accumulate_cost_matrix(C):
    """
    Build the accumulated cost matrix D of size (N+1 x M+1) 
    using small trick for simplifying the initialization:
    set:
        D[0, 0] = 0
        D[n, 0] = inf  for n >= 1
        D[0, m] = inf  for m >= 1
    for all n in [1..N] and m in [1..M]:
        D[i, j] = C[i, j] + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    
    Args:
        numpy array of shape (N, M) — the local cost matrix
 
    Returns:
        numpy array of shape (N+1, M+1) — the accumulated cost matrix
    """
    N, M = C.shape
    D = np.full((N + 1, M + 1), np.inf)
    D[0, 0] = 0.0
 
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            D[i, j] = C[i - 1, j - 1] + min(
                D[i - 1, j], # vertical step
                D[i, j - 1], # horizontal step
                D[i - 1, j - 1]) # diagonal step
 
    return D

def backtrack_warping_path(D):
    """
    Backtrack through the D to find the optimal warping path P.
 
    Args:
        D: numpy array of shape (N+1, M+1) — the accumulated cost matrix
 
    Returns:
        list of tuples — the optimal warping path as a list of (i, j) indices
        ordered from the start (0, 0) to the end (N-1, M-1)
    """
    N = D.shape[0] - 1
    M = D.shape[1] - 1

    path = []
    
    while N > 0 and M > 0:
        path.append((N-1, M-1))
        # Find the minimum cost step
        diag = D[N-1, M-1]
        vertical = D[N-1, M  ]
        horizontal = D[N, M-1]
        min_step = min(diag, vertical, horizontal)
        if min_step == vertical:
            N -= 1
        elif min_step == horizontal:
            M -= 1
        else: # diagonal step
            N -= 1
            M -= 1
 
    # Reverse to get the path from start to end
    path.reverse()  
    return path


def note_alignment_DTW(response_notes, ref_notes):
    """
    DTW pipeline: build cost matrix -> build accumulated cost matrix -> backtrack
 
    Args:
        response_notes: The student's response MIDI notes to evaluate
        ref_notes: The reference MIDI note
 
    Returns:
        path: list of (response_idx, ref_idx) pairs — the optimal alignment
        C: local cost matrix (useful for visualisation)
        D: accumulated cost matrix (useful for visualisation)
    """
    C = cost_matrix(response_notes, ref_notes)
    D = accumulate_cost_matrix(C)
    path = backtrack_warping_path(D)
    return path, C, D

Evaluation

In [8]:
def evaluate_note_pair(response_note, ref_note, ref_idx,
                       timing_tolerance=0.1, duration_tolerance=0.1):
    """
    Evaluate a single aligned note pair and return feedback.
 
    Args:
        response_note: student's note dict
        ref_note: reference note dict
        ref_idx: 1-based display index (based on ref position)
        timing_tolerance: consider as correct if start is within this tolerance
        duration_tolerance: consider as correct if duration is within this tolerance
 
    Returns:
        is_correct (bool), messages (list of str)
    """
    messages = []
    is_correct = True
 
    # Pitch check
    if response_note["pitch"] != ref_note["pitch"]:
        is_correct = False
        messages.append(
            f"Note {ref_idx}: wrong pitch — expected {ref_note['pitch']}, "
            f"played {response_note['pitch']}."
        )
 
    # Timing check
    timing_diff = abs(response_note["start"] - ref_note["start"])
    if timing_diff > timing_tolerance:
        is_correct = False
        messages.append(f"Note {ref_idx}: difference in start time: {timing_diff:.2f}s.")
 
    # Duration check
    duration_diff = abs(response_note["duration"] - ref_note["duration"])
    if duration_diff > duration_tolerance:
        is_correct = False
        messages.append(f"Note {ref_idx}: difference in duration: {duration_diff:.2f}s.")
 
    if is_correct:
        messages.append(f"Note {ref_idx} (with pitch {ref_note['pitch']}) is correct.")
 
    return is_correct, messages

In [9]:
def comparison(response, ref,
                   timing_tolerance=0.1, duration_tolerance=0.1):
    """
    Compare student MIDI against reference MIDI after DTW-based alignment.
 
    Args:
        response: The student's response MIDI
        ref: The reference MIDI
        timing_tolerance:   seconds
        duration_tolerance: seconds
 
    Returns:
        all_correct (bool), feedback (list of str)
    """
    response_notes = response["notes"]
    ref_notes = ref["notes"]
 
    # Align using DTW — response first, ref second
    path, C, D = note_alignment_DTW(response_notes, ref_notes)
 
    feedback = []
    all_correct = True
    
    for response_idx, ref_idx in path:
        is_correct, messages = evaluate_note_pair(
            response_notes[response_idx], ref_notes[ref_idx],
            ref_idx=ref_idx + 1,
            timing_tolerance=timing_tolerance,
            duration_tolerance=duration_tolerance,
        )
        if not is_correct:
            all_correct = False
        feedback.extend(messages)
 
    return all_correct, feedback

In [10]:
def evaluation_function(response: Any, answer: Any, params: Params) -> Result:
    """
    Entry point for Lambda Feedback.
 
    Args:
        response: student MIDI dict
        answer:   reference MIDI dict
        params:   optional extra parameters
 
    Returns:
        Result with is_correct and feedback string
    """
    all_correct, feedback = comparison(response, answer)
    return Result(
        is_correct=all_correct,
        feedback_items=[("feedback", "\n".join(feedback))]
    )

In [12]:
is_correct, feedbacks = comparison(
    response,
    reference
)

print(is_correct)

for feedback in feedbacks:
    print(feedback)

False
Note 1 (with pitch 60) is correct.
Note 2: wrong pitch — expected 62, played 63.
Note 3: difference in start time: 0.15s.
Note 4: difference in duration: 0.20s.
Note 5: wrong pitch — expected 67, played 65.
Note 5: difference in start time: 0.70s.
Note 5: difference in duration: 0.20s.


note5 should be a missing pitch!! Need to check the DTW algorithm!